In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import psycopg2
import json, os, glob
import pyarrow.parquet as pq

In [ ]:
# ------------------------------
# Postgres connection
# ------------------------------
user = "ethel"
password = "pa55w0rd13"  # replace with your Postgres password
host = "localhost"
port = "5432"
database = "swimming_pools_db"

engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}")

In [ ]:
# ------------------------------
# Allow place_id to be nullable in places
# ------------------------------
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE places
        ALTER COLUMN place_id DROP NOT NULL;
    """))

    print("Updated places table: place_id is now nullable; unique_identifier remains the required key.")

In [ ]:
# ------------------------------
# Add missing columns to reviews table
# ------------------------------
with engine.begin() as conn:
    # Add retrieved_time column (timestamp)
    conn.execute(text("""
        ALTER TABLE reviews 
        ADD COLUMN IF NOT EXISTS retrieved_time TIMESTAMP;
    """))
    
    # Add other_ratings column (integer or text, depending on your data)
    conn.execute(text("""
        ALTER TABLE reviews 
        ADD COLUMN IF NOT EXISTS other_ratings INTEGER;
    """))
    
    print("Added columns to reviews table:")
    print("  - retrieved_time (TIMESTAMP)")
    print("  - other_ratings (INTEGER)")

## the places info

In [ ]:
# ------------------------------
# Path to Parquet
# ------------------------------
#parquet_file = '/home/ethel/Desktop/swimming_pool_project/clean_datasets/germany_places.parquet'
folder_path = '/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany'
file_pattern = os.path.join(folder_path, '*_gpi.parquet')

parquet_files = sorted(glob.glob(file_pattern))

if not parquet_files:
    print(f"No files found for pattern: {file_pattern}")
# ------------------------------
# Columns that are JSONB
# ------------------------------
jsonb_columns = [
    "place_subtype", "opening_hours", "service_options", "higlights",
    "popular_for", "accessibility", "offerings", "dining_options",
    "amenities", "atmosphere", "crowd_level", "planning",
    "payment_options", "children", "parking"
]

import ast

def to_jsonb_value(value):
    if value is None:
        return None
    if isinstance(value, (list, dict)):
        if isinstance(value, list) and len(value) == 1 and isinstance(value[0], str):
            inner = value[0].strip()
            if inner.startswith('[') and inner.endswith(']'):
                try:
                    parsed_inner = ast.literal_eval(inner)
                    if isinstance(parsed_inner, (list, tuple, set)):
                        return json.dumps(list(parsed_inner))
                except (ValueError, SyntaxError):
                    pass
        return json.dumps(value)
    if hasattr(value, 'tolist') and not isinstance(value, str):
        return to_jsonb_value(value.tolist())
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = str(value).strip()
    if not text:
        return None
    if len(text) >= 2 and ((text[0] == text[-1] == '"') or (text[0] == text[-1] == "'")):
        text = text[1:-1].strip()

    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return json.dumps(value)

    if isinstance(parsed, str):
        return to_jsonb_value(parsed)
    if isinstance(parsed, (list, tuple, set)):
        if len(parsed) == 1 and isinstance(next(iter(parsed)), str):
            single_item = next(iter(parsed)).strip()
            if single_item.startswith('[') and single_item.endswith(']'):
                try:
                    parsed_inner = ast.literal_eval(single_item)
                    if isinstance(parsed_inner, (list, tuple, set)):
                        return json.dumps(list(parsed_inner))
                except (ValueError, SyntaxError):
                    pass
        return json.dumps(list(parsed))
    if isinstance(parsed, dict):
        return json.dumps(parsed)
    return json.dumps(parsed)

for parquet_file in parquet_files:
    print(f"Processing file: {parquet_file}")
    # ------------------------------
    # Pre-check: find columns with at least one non-null value across full file
    # ------------------------------
    pf_full = pq.read_table(parquet_file).to_pandas()
    cols_to_keep = pf_full.columns[pf_full.notnull().any()].tolist()
    print(f"Keeping {len(cols_to_keep)} columns, dropped: {set(pf_full.columns) - set(cols_to_keep)}")
    del pf_full  # free memory

    # ------------------------------
    # Load in chunks
    # ------------------------------
    chunksize = 5000
    pf = pq.ParquetFile(parquet_file)

    for batch in pf.iter_batches(batch_size=chunksize):
        chunk = batch.to_pandas()

        # Drop all-null columns (consistent across batches)
        chunk = chunk[cols_to_keep]

        # Skip rows that cannot satisfy the ON CONFLICT target
        if 'unique_identifier' in chunk.columns:
            chunk = chunk[chunk['unique_identifier'].notna() & (chunk['unique_identifier'] != '')]

        if len(chunk) == 0:
            continue

        # Replace float NaN with None so SQLAlchemy sends NULL
        chunk = chunk.where(pd.notnull(chunk), other=None)

        # Convert JSONB columns to JSON strings
        for col in jsonb_columns:
            if col in chunk.columns:
                chunk[col] = chunk[col].apply(to_jsonb_value)

        # Create WKT geometry from lat/lon
        chunk["geom"] = chunk.apply(
            lambda row: f"SRID=4326;POINT({row['longitude']} {row['latitude']})"
            if row["latitude"] is not None and row["longitude"] is not None else None,
            axis=1
        )

        # Build upsert query
        insert_columns = list(chunk.columns)
        insert_cols_str = ",".join(insert_columns)
        values_placeholders = ",".join([f":{col}" for col in insert_columns])

        update_cols = [col for col in insert_columns if col != "unique_identifier"]
        update_str = ", ".join([f"{col}=EXCLUDED.{col}" for col in update_cols])

        upsert_query = f"""
            INSERT INTO places ({insert_cols_str})
            VALUES ({values_placeholders})
            ON CONFLICT (unique_identifier) DO UPDATE
            SET {update_str};
        """

        records = chunk.to_dict(orient="records")
        # Final safety: replace any remaining nan/NA in jsonb columns with None
        for record in records:
            for col in jsonb_columns:
                if col in record:
                    record[col] = to_jsonb_value(record[col])

        # Execute upsert for this chunk immediately so all batches are persisted
        with engine.begin() as conn:
            conn.execute(text(upsert_query), records)

    print("Upsert complete!")

In [ ]:
# ------------------------------
# Inspect the places table (counts NULLs and empty/whitespace for text columns)
# ------------------------------
import pandas as pd

inspection_query = """
    SELECT
        COUNT(*) AS row_count
    FROM places;
"""

with engine.connect() as conn:
    # total rows
    row_count_df = pd.read_sql(text(inspection_query), conn)
    total_rows = int(row_count_df.loc[0, 'row_count'])

    # get columns and data types
    columns_df = pd.read_sql(text("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = 'places'
        ORDER BY ordinal_position;
    """), conn)

    def compute_null_counts(conn, columns_df):
        rows = []
        for _, r in columns_df.iterrows():
            col = r['column_name']
            dtype = r['data_type']
            if dtype in ('text', 'character varying', 'varchar', 'char'):
                q = text(f"SELECT COUNT(*) FROM places WHERE ({col} IS NULL OR TRIM(COALESCE({col}, '')) = '')")
            else:
                q = text(f"SELECT COUNT(*) FROM places WHERE {col} IS NULL")
            null_count = conn.execute(q).scalar_one()
            rows.append({'column_name': col, 'null_rows': int(null_count)})
        df_nulls = pd.DataFrame(rows)
        df_nulls['non_null_rows'] = total_rows - df_nulls['null_rows']
        df_nulls['null_pct'] = (df_nulls['null_rows'] / total_rows * 100).round(2) if total_rows else 0
        return df_nulls

    # compute current null/empty counts (single table)
    nulls_df = compute_null_counts(conn, columns_df)

    print(f'Total rows in places: {total_rows}')
    display(nulls_df.sort_values(['non_null_rows', 'column_name'], ascending=[False, True]))

print('Inspection complete.')

In [ ]:
# ------------------------------
# Read parquet files in chunks and upsert into Postgres (with inline cleanup)
# ------------------------------
import pyarrow.parquet as pq
import pandas as pd

for parquet_file in parquet_files:
    pf = pq.ParquetFile(parquet_file)
    print(f"Processing: {parquet_file}")

    for batch in pf.iter_batches(batch_size=chunksize):
        chunk = batch.to_pandas()

        # --- Inline cleanup: trim text columns and convert empty/whitespace-only to NULL ---
        str_cols = chunk.select_dtypes(include=['object']).columns.tolist()
        for c in str_cols:
            chunk[c] = chunk[c].apply(lambda x: x.strip() if isinstance(x, str) else x)
            chunk[c] = chunk[c].apply(lambda x: None if x is None or (isinstance(x, str) and x == '') else x)

        # Ensure identifier columns are normalized (explicitly handle place_id)
        if 'place_id' in chunk.columns:
            chunk['place_id'] = chunk['place_id'].apply(lambda x: None if x is None or (isinstance(x, str) and x == '') else x)

        # --- JSONB-safe serialization for json-like columns ---
        for col in jsonb_columns:
            if col in chunk.columns:
                chunk[col] = chunk[col].apply(to_jsonb_value)

        # Replace pandas NA with Python None so DB gets proper NULLs
        chunk = chunk.where(pd.notnull(chunk), None)

        # Convert to list of dicts for executemany
        records = chunk.to_dict(orient='records')

        # Upsert into DB using previously-defined upsert query
        with engine.begin() as conn:
            if records:
                conn.execute(text(upsert_query), records)

    print(f"Finished file: {parquet_file}")

print('All parquet files processed.')

## the reviews info

In [ ]:
# parquet_file = '/home/ethel/Desktop/swimming_pool_project/clean_datasets/merged_reviews.parquet'
#parquet_file = '/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/germany_reviews.parquet'

jsonb_columns = ["other_reviews", "uploaded_pics", "uploaded_pics_links"]
folder_path = '/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/'
file_pattern = os.path.join(folder_path, '*reviews.parquet')

parquet_files = sorted(glob.glob(file_pattern))

if not parquet_files:
    print(f"No files found for pattern: {file_pattern}")

for parquet_file in parquet_files:
    chunksize = 5000
    pf = pq.ParquetFile(parquet_file)
    # Columns that exist in the DB table (drop anything else)
    db_columns = [
        "place_id", "unique_identifier", "retrieved_time", "reviewer_id",
        "reviewer_name", "review_id", "review_link", "contributor_link",
        "review_time", "org_review", "org_review_lang", "trans_review",
        "reviewer_rating", "other_reviews", "other_ratings", "uploaded_pics",
        "profile_pic", "review_location", "uploaded_pics_links"
    ]

    # Build a lookup so rows with missing unique_identifier can be recovered from places via place_id
    with engine.connect() as conn:
        place_map_df = pd.read_sql(
            text("""
                SELECT place_id, unique_identifier
                FROM places
                WHERE place_id IS NOT NULL
                AND TRIM(COALESCE(place_id, '')) <> ''
                AND unique_identifier IS NOT NULL
                AND TRIM(COALESCE(unique_identifier, '')) <> ''
            """),
            conn,
        )

    place_id_to_uid = (
        place_map_df.dropna(subset=["place_id", "unique_identifier"])
        .drop_duplicates(subset=["place_id"])
        .set_index("place_id")["unique_identifier"]
        .to_dict()
    )
    print(f"Loaded {len(place_id_to_uid)} place_id -> unique_identifier mappings from places")

    batch_count = 0
    total_rows = 0
    recovered_uid_count = 0
    unresolved_uid_count = 0
    skipped_review_id_count = 0

    for batch in pf.iter_batches(batch_size=chunksize):
        df = batch.to_pandas()
        # Drop columns not in the DB
        df = df[[col for col in db_columns if col in df.columns]]

        # Normalize text fields before key recovery/filtering
        str_cols = df.select_dtypes(include=["object", "str"]).columns
        for col in str_cols:
            df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

        # Recover missing unique_identifier from places using place_id
        if "place_id" in df.columns and "unique_identifier" in df.columns:
            missing_uid_mask = df["unique_identifier"].isna() | (df["unique_identifier"] == "")
            recoverable_mask = missing_uid_mask & df["place_id"].notna() & (df["place_id"] != "")
            if recoverable_mask.any():
                recovered_values = df.loc[recoverable_mask, "place_id"].map(place_id_to_uid)
                df.loc[recoverable_mask, "unique_identifier"] = recovered_values
                recovered_uid_count += int(recovered_values.notna().sum())

        # Keep rows that now have a resolvable unique_identifier
        unresolved_mask = df["unique_identifier"].isna() | (df["unique_identifier"] == "")
        unresolved_uid_count += int(unresolved_mask.sum())
        df = df[~unresolved_mask]
        # review_id is the ON CONFLICT key and must never be null/blank
        review_id_mask = df["review_id"].notna() & (df["review_id"] != "")
        skipped_review_id_count += int((~review_id_mask).sum())
        df = df[review_id_mask]

        if len(df) == 0:
            print(f"Batch {batch_count + 1}: All rows skipped (no resolvable unique_identifier)")
            batch_count += 1
            continue

        # Replace all NaN/NA with None
        df = df.where(pd.notnull(df), other=None)

        # Convert JSONB columns
        for col in jsonb_columns:
            if col in df.columns:
                # Convert Arrow-backed column to plain Python objects first
                df[col] = df[col].astype(object)
                df[col] = df[col].apply(lambda x: json.dumps(x) if not pd.isna(x) else None)


        # Build upsert query
        insert_columns = list(df.columns)
        insert_cols_str = ",".join(insert_columns)
        values_placeholders = ",".join([f":{col}" for col in insert_columns])
        update_cols = [col for col in insert_columns if col != "review_id"]
        update_str = ", ".join([f"{col}=EXCLUDED.{col}" for col in update_cols])

        upsert_query = f"""
            INSERT INTO reviews ({insert_cols_str})
            VALUES ({values_placeholders})
            ON CONFLICT (review_id) DO UPDATE
            SET {update_str};
        """

        records = df.to_dict(orient="records")

        for record in records:
            for col, val in record.items():
                # Strip NUL bytes from strings
                if isinstance(val, str):
                    record[col] = val.replace("\x00", "")
                # Replace any remaining nan/NA with None (catches jsonb columns too)
                elif val is not None:
                    try:
                        if pd.isna(val):
                            record[col] = None
                    except (TypeError, ValueError):
                        pass

        # Execute upsert for this chunk immediately so all batches are persisted
        with engine.begin() as conn:
            conn.execute(text(upsert_query), records)

        batch_count += 1
        total_rows += len(df)
        print(f"Processed batch {batch_count} ({len(df)} rows, total: {total_rows})")

    print(
        f"Reviews upsert complete! Total batches: {batch_count}, Total rows: {total_rows}, "
        f"Recovered unique_identifier values: {recovered_uid_count}, "
        f"Still-unresolved rows skipped: {unresolved_uid_count}, "
        f"Rows skipped for missing review_id: {skipped_review_id_count}"
    )

In [ ]:
# ------------------------------
# Export unmatched review place_ids and inspect the reviews table
# ------------------------------
import os
import pandas as pd

reviews_parquet_file = '/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/processed_data/final_all_reviews_updated_30042026.parquet'
output_dir = '/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data'
os.makedirs(output_dir, exist_ok=True)

with engine.connect() as conn:
    places_lookup_df = pd.read_sql(
        text("""
            SELECT DISTINCT place_id
            FROM places
            WHERE place_id IS NOT NULL
              AND TRIM(COALESCE(place_id, '')) <> ''
        """),
        conn,
    )
    review_row_count_df = pd.read_sql(text("SELECT COUNT(*) AS row_count FROM reviews;"), conn)
    reviews_total_rows = int(review_row_count_df.loc[0, 'row_count'])
    reviews_columns_df = pd.read_sql(text("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = 'reviews'
        ORDER BY ordinal_position;
    """), conn)

places_lookup = set(places_lookup_df['place_id'].astype(str))

unmatched_rows = []
reviews_pf = pq.ParquetFile(reviews_parquet_file)
for batch in reviews_pf.iter_batches(batch_size=5000):
    df = batch.to_pandas()
    if 'place_id' not in df.columns or 'unique_identifier' not in df.columns:
        continue

    str_cols = df.select_dtypes(include=['object', 'str']).columns
    for col in str_cols:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

    missing_uid_mask = df['unique_identifier'].isna() | (df['unique_identifier'] == '')
    unmatched_mask = (
        missing_uid_mask
        & df['place_id'].notna()
        & (df['place_id'] != '')
        & ~df['place_id'].isin(places_lookup)
    )

    if unmatched_mask.any():
        cols_to_keep = [col for col in ['place_id', 'review_id', 'reviewer_id', 'reviewer_name', 'retrieved_time'] if col in df.columns]
        unmatched_rows.append(df.loc[unmatched_mask, cols_to_keep])

if unmatched_rows:
    unmatched_df = pd.concat(unmatched_rows, ignore_index=True)
else:
    unmatched_df = pd.DataFrame(columns=['place_id', 'review_id', 'reviewer_id', 'reviewer_name', 'retrieved_time'])

unmatched_summary_df = (
    unmatched_df.groupby('place_id', as_index=False)
    .size()
    .rename(columns={'size': 'unmatched_review_rows'})
    .sort_values(['unmatched_review_rows', 'place_id'], ascending=[False, True])
)

csv_path = os.path.join(output_dir, 'unmatched_review_place_ids.csv')
unmatched_summary_df.to_csv(csv_path, index=False)

print(f"Unmatched review rows: {len(unmatched_df)}")
print(f"Distinct unmatched place_id values: {len(unmatched_summary_df)}")
print(f"Exported unmatched place_id summary to: {csv_path}")
print("\nReviews table row count and null/empty counts:")

def compute_review_null_counts(conn, columns_df):
    rows = []
    for _, r in columns_df.iterrows():
        col = r['column_name']
        dtype = r['data_type']
        if dtype in ('text', 'character varying', 'varchar', 'char'):
            q = text(f"SELECT COUNT(*) FROM reviews WHERE ({col} IS NULL OR TRIM(COALESCE({col}, '')) = '')")
        else:
            q = text(f"SELECT COUNT(*) FROM reviews WHERE {col} IS NULL")
        null_count = conn.execute(q).scalar_one()
        rows.append({'column_name': col, 'null_rows': int(null_count)})
    df_nulls = pd.DataFrame(rows)
    df_nulls['non_null_rows'] = reviews_total_rows - df_nulls['null_rows']
    df_nulls['null_pct'] = (df_nulls['null_rows'] / reviews_total_rows * 100).round(2) if reviews_total_rows else 0
    return df_nulls

with engine.connect() as conn:
    reviews_nulls_df = compute_review_null_counts(conn, reviews_columns_df)
    print(f'Total rows in reviews: {reviews_total_rows}')
    display(reviews_nulls_df.sort_values(['non_null_rows', 'column_name'], ascending=[False, True]))

print('Reviews inspection complete.')

### Create reviews_details table

## reviews details info

In [ ]:
# ------------------------------
# Create reviews_details table
# ------------------------------
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS reviews_details (
            place_id TEXT NOT NULL,
            unique_identifier TEXT NOT NULL REFERENCES places(unique_identifier) ON DELETE CASCADE,
            retrieved_time TIMESTAMP,
            json_url TEXT,
            next_key TEXT,
            extracted_reviews INTEGER,
            PRIMARY KEY (place_id, unique_identifier, retrieved_time)
        );
    """))
    
    print("✓ Table 'reviews_details' created successfully!")
    
# Check the table structure
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT column_name, data_type, is_nullable
        FROM information_schema.columns
        WHERE table_name = 'reviews_details'
        ORDER BY ordinal_position;
    """))
    
    print("\nColumns in 'reviews_details' table:")
    print(f"{'Column Name':<30} {'Data Type':<30} {'Nullable':<10}")
    print("-" * 70)
    for row in result:
        print(f"{row[0]:<30} {row[1]:<30} {row[2]:<10}")

In [ ]:
import os
import glob

# ------------------------------
# Insert data from *_details.parquet files into reviews_details
# ------------------------------
# folder_path = '/home/ethel/Desktop/swimming_pool_project/more_reviews_json'
# file_pattern = os.path.join(folder_path, '*_details.parquet')
folder_path = '/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/'
file_pattern = os.path.join(folder_path, '*_details.parquet')

parquet_files = sorted(glob.glob(file_pattern))

if not parquet_files:
    print(f"No files found for pattern: {file_pattern}")

for parquet_file in parquet_files:
    print(f"Processing file: {parquet_file}")

    chunksize = 5000
    pf = pq.ParquetFile(parquet_file)

    batch_count = 0
    total_rows = 0

    for batch in pf.iter_batches(batch_size=chunksize):
        df = batch.to_pandas()

        # Keep only rows that can match the table key
        df = df[df['unique_identifier'].notna()]
        if len(df) == 0:
            batch_count += 1
            continue

        # Convert and drop rows where retrieved_time is NaT (part of conflict key)
        if 'retrieved_time' in df.columns:
            df['retrieved_time'] = pd.to_datetime(
                df['retrieved_time'],
                format='%Y-%m-%d %H:%M:%S',
                errors='coerce'
            )
            df = df[df['retrieved_time'].notna()]

        if len(df) == 0:
            batch_count += 1
            continue

        insert_columns = list(df.columns)
        insert_cols_str = ",".join(insert_columns)
        values_placeholders = ",".join([f":{col}" for col in insert_columns])
        update_cols = [
            col for col in insert_columns
            if col not in ["place_id", "unique_identifier", "retrieved_time"]
        ]
        update_str = ", ".join([f"{col}=EXCLUDED.{col}" for col in update_cols])

        upsert_query = f"""
            INSERT INTO reviews_details ({insert_cols_str})
            VALUES ({values_placeholders})
            ON CONFLICT (place_id, unique_identifier, retrieved_time) DO UPDATE
            SET {update_str};
        """

        records = df.to_dict(orient="records")
        for record in records:
            for col, val in record.items():
                if isinstance(val, str):
                    record[col] = val.replace("\x00", "")
                elif val is not None:
                    try:
                        if pd.isna(val):
                            record[col] = None
                    except (TypeError, ValueError):
                        pass

        # Try to execute; skip FK violations gracefully
        try:
            with engine.begin() as conn:
                conn.execute(text(upsert_query), records)
        except Exception as err:
            if "ForeignKeyViolation" in str(type(err).__name__) or "foreign key" in str(err).lower():
                print(f"  ⚠ Batch skipped (FK violation for missing places)")
            else:
                raise

        batch_count += 1
        total_rows += len(df)
        print(f"Processed batch {batch_count} ({len(df)} rows, total: {total_rows})")

    print(f"Finished file: {parquet_file} | batches: {batch_count}, rows: {total_rows}")

### Getting the last next key

In [ ]:
# ------------------------------
# Export review_count + latest next_key per place to CSV
# ------------------------------
import os

query = """
    WITH review_counts AS (
        SELECT
            place_id,
            COALESCE(unique_identifier, place_id) AS unique_identifier,
            COUNT(DISTINCT review_id) AS review_count
        FROM reviews
        WHERE review_id IS NOT NULL
        GROUP BY place_id, COALESCE(unique_identifier, place_id)
    ),
    latest_next_key AS (
        SELECT
            place_id,
            COALESCE(unique_identifier, place_id) AS unique_identifier,
            next_key,
            ROW_NUMBER() OVER (
                PARTITION BY COALESCE(unique_identifier, place_id)
                ORDER BY retrieved_time DESC NULLS LAST
            ) AS rn
        FROM reviews_details
    )
    SELECT
        rc.place_id,
        rc.unique_identifier,
        rc.review_count,
        lnk.next_key
    FROM review_counts rc
    LEFT JOIN latest_next_key lnk
        ON rc.unique_identifier = lnk.unique_identifier
       AND lnk.rn = 1
    ORDER BY rc.review_count DESC, rc.unique_identifier;
"""

with engine.connect() as conn:
    result = conn.execute(text(query))
    df_export = pd.DataFrame(
        result.fetchall(),
        columns=['place_id', 'unique_identifier', 'review_count', 'next_key']
    )

print(f"Fetched rows: {len(df_export)}")
print(df_export.head(10))

output_dir = '/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data'
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, 'review_counts_with_latest_next_key.csv')
df_export.to_csv(csv_path, index=False)

print(f"CSV exported to: {csv_path}")


In [ ]:
dff = pd.read_parquet("/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/reviews_flagged.parquet")
dff.info()